In [66]:
# Cell 1: Setup & imports
"""
Data Download Exploration - Framework_V1
=========================================
Goal: Understand how Upstox API works and download TATASTEEL data
"""

import upstox_client
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import time
import calendar

print("✅ Imports successful")
print(f"📦 pandas Version: {pd.__version__}")

✅ Imports successful
📦 pandas Version: 2.3.3


In [67]:
# Cell 2: configure Upstox API

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get token from .env
ACCESS_TOKEN = os.getenv('UPSTOX_ACCESS_TOKEN')

if not ACCESS_TOKEN:
    raise ValueError("❌ UPSTOX_ACCESS_TOKEN not found in .env file")

# Test instrument: TATASTEEL
TATASTEEL_KEY = 'NSE_EQ|INE081A01020'

# Setup API configuration
configuration = upstox_client.Configuration()
configuration.access_token = ACCESS_TOKEN

print("✅ API configured")
print(f"📊 Test instrument: TATASTEEL")
print(f"🔑 Instrument key: {TATASTEEL_KEY}")

✅ API configured
📊 Test instrument: TATASTEEL
🔑 Instrument key: NSE_EQ|INE081A01020


In [68]:
# Cell 3: Test API Connection - Download 1 day of data

def fetch_data(instrument_key, from_date, to_date, interval='5', unit="minutes"):
    """Fetch historical data from Upstox"""
    try:
        api_instance = upstox_client.HistoryV3Api(upstox_client.ApiClient(configuration))
        api_response = api_instance.get_historical_candle_data1(
            instrument_key=instrument_key,
            unit=unit,
            interval=interval,
            to_date=to_date.strftime('%Y-%m-%d'),
            from_date=from_date.strftime('%Y-%m-%d')
        )

        if not hasattr(api_response, 'data') or not api_response.data:
            return None

        candles = api_response.data.candles
        if len(candles) == 0:
            return None

        df = pd.DataFrame(candles, columns=['datetime', 'open', 'high', 'low', 'close', 'volume', 'oi'])
        df['datetime'] = pd.to_datetime(df['datetime'])
        df = df.sort_values('datetime').reset_index(drop=True)

        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Test: Download 1 Day (find last trading day)
def find_last_trading_day(instrument_key, max_days_back=7):
    """Find the last trading day with available data"""
    for days_back in range(1, max_days_back + 1):
        test_date = datetime.now() - timedelta(days=days_back)
        df = fetch_data(instrument_key, test_date, test_date)
        if df is not None and len(df) > 0:
            return test_date, df
    return None, None

test_data, df_test = find_last_trading_day(TATASTEEL_KEY)

if df_test is not None:
    print(f"✅ API connection successful!")
    print(f"📊 Downloaded {len(df_test)} candles for {test_data.strftime('%Y-%m-%d')}")
    print(f"\nFirst 3 rows:")
    display(df_test.head(3))
else:
    print(f"❌ API connection failed or no data available in last 7 days")

✅ API connection successful!
📊 Downloaded 75 candles for 2026-02-06

First 3 rows:


,datetime,open,high,low,close,volume,oi
0,2026-02-06 09:15:00+05:30,197.60,197.60,195.10,196.48,930832,0
1,2026-02-06 09:20:00+05:30,196.50,197.00,196.31,196.33,507748,0
2,2026-02-06 09:25:00+05:30,196.37,196.65,195.93,196.45,474699,0


In [69]:
# Cell 4: Examine DataFrame Structure

print("📋 DataFrame Info:")
print(df_test.info())

print("\n📊 Data Statistics:")
display(df_test.describe())

print(f"\n🕐 Time Range:")
print(f"   Start: {df_test['datetime'].min()}")
print(f"   End:   {df_test['datetime'].max()}")

print(f"\n💹 Price Range:")
print(f"   High: ₹{df_test['high'].max():.2f}")
print(f"   Low:  ₹{df_test['low'].min():.2f}")

📋 DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype                    
---  ------    --------------  -----                    
 0   datetime  75 non-null     datetime64[ns, UTC+05:30]
 1   open      75 non-null     float64                  
 2   high      75 non-null     float64                  
 3   low       75 non-null     float64                  
 4   close     75 non-null     float64                  
 5   volume    75 non-null     int64                    
 6   oi        75 non-null     int64                    
dtypes: datetime64[ns, UTC+05:30](1), float64(4), int64(2)
memory usage: 4.2 KB
None

📊 Data Statistics:


,open,high,low,close,volume,oi
count,75.000000,75.000000,75.000000,75.000000,75.000000,75.0
mean,196.008800,196.236667,195.750933,195.990667,245929.786667,0.0
std,0.670826,0.645697,0.658436,0.642130,184126.022879,0.0
min,194.600000,194.990000,194.370000,194.660000,35297.000000,0.0
25%,195.490000,195.775000,195.220000,195.485000,116353.000000,0.0
50%,196.100000,196.250000,195.770000,196.120000,186422.000000,0.0
75%,196.420000,196.655000,196.140000,196.425000,347770.000000,0.0
max,197.600000,197.600000,197.120000,197.350000,930832.000000,0.0



🕐 Time Range:
   Start: 2026-02-06 09:15:00+05:30
   End:   2026-02-06 15:25:00+05:30

💹 Price Range:
   High: ₹197.60
   Low:  ₹194.37


In [70]:
#Cell 5: Download 1 Month of Data (Jan 2024)

start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 31)

print(f"📥 Downloading TATASTEEL 5-min data...")
print(f"   Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")

df_month = fetch_data(TATASTEEL_KEY, start_date, end_date)

if df_month is not None:
    print(f"✅ Downloaded {len(df_month)} candles")
    print(f"\n📊 Sample data:")
    display(df_month.head(10))
else:
    print("❌ Download failed")

📥 Downloading TATASTEEL 5-min data...
   Period: 2024-01-01 to 2024-01-31
✅ Downloaded 1650 candles

📊 Sample data:


,datetime,open,high,low,close,volume,oi
0,2024-01-01 09:15:00+05:30,136.40,136.40,135.45,136.20,974407,0
1,2024-01-01 09:20:00+05:30,136.20,136.20,135.75,135.85,632912,0
2,2024-01-01 09:25:00+05:30,135.85,136.15,135.55,136.15,534765,0
3,2024-01-01 09:30:00+05:30,136.15,136.15,135.70,135.85,394908,0
4,2024-01-01 09:35:00+05:30,135.80,136.20,135.80,136.00,459369,0
5,2024-01-01 09:40:00+05:30,136.00,136.15,135.85,136.05,441722,0
6,2024-01-01 09:45:00+05:30,136.05,136.30,136.00,136.30,268912,0
7,2024-01-01 09:50:00+05:30,136.25,136.55,136.25,136.40,705535,0
8,2024-01-01 09:55:00+05:30,136.40,136.55,136.30,136.55,317530,0
9,2024-01-01 10:00:00+05:30,136.50,137.20,136.50,137.10,1173917,0


In [71]:
# Cell 6: Check for Missing Dates/Gaps

# Expected: ~75 candles per trading day(375 minutes / 5-min candles)
# Jan 2024 had ~22 trading days = ~1650 candles expected

trading_days = df_month.groupby(df_month['datetime'].dt.date).size()

print(f"📅 Trading Days in Dataset: {len(trading_days)}")
print(f"📊 Candles per Day:")
display(trading_days)

print(f"\n📈 Total Candles: {len(df_month)}")
print(f"📊 Avg Candles/Day: {len(df_month) / len(trading_days):.1f}")
print(f"✅ Expected: ~75 candles/day (9:15 AM - 3:30 PM")

📅 Trading Days in Dataset: 22
📊 Candles per Day:


datetime
2024-01-01    75
2024-01-02    75
2024-01-03    75
2024-01-04    75
2024-01-05    75
2024-01-08    75
2024-01-09    75
2024-01-10    75
2024-01-11    75
2024-01-12    75
2024-01-15    75
2024-01-16    75
2024-01-17    75
2024-01-18    75
2024-01-19    75
2024-01-20    75
2024-01-23    75
2024-01-24    75
2024-01-25    75
2024-01-29    75
2024-01-30    75
2024-01-31    75
dtype: int64


📈 Total Candles: 1650
📊 Avg Candles/Day: 75.0
✅ Expected: ~75 candles/day (9:15 AM - 3:30 PM


In [83]:
# Inspect parquet schema (shows true stored types)
import pyarrow.parquet as pq

table = pq.read_table(test_path)
print(table.schema)

# Cell 7: Test Parquet Save/Load

# Define path
test_path = Path("../data/historical/intraday_5min/TATASTEEL_test.parquet")
test_path.parent.mkdir(parents=True, exist_ok=True)

# Save
df_month["datetime"] = (pd.to_datetime(df_month["datetime"]).dt.strftime("%Y-%m-%d %H:%M:%S"))
df_month.to_parquet(test_path, index=False)
print(f"💾 Saved: {test_path}")
print(f"📦 File size: {test_path.stat().st_size / 1024:.1f} KB")

# Load back
df_loaded = pd.read_parquet(test_path)
print(f"\n✅ Loaded: {len(df_loaded)} rows")
print(f"🔍 Data integrity check: {df_month.equals(df_loaded)}")

# Show loaded data
display(df_loaded.head(3))

print("Pandas dtype:")
print(df_month.dtypes)

print("\nParquet schema:")
print(table.schema)


datetime: string
open: double
high: double
low: double
close: double
volume: int64
oi: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 855
💾 Saved: ..\data\historical\intraday_5min\TATASTEEL_test.parquet
📦 File size: 33.9 KB

✅ Loaded: 1650 rows
🔍 Data integrity check: True


,datetime,open,high,low,close,volume,oi
0,2024-01-01 09:15:00,136.40,136.40,135.45,136.20,974407,0
1,2024-01-01 09:20:00,136.20,136.20,135.75,135.85,632912,0
2,2024-01-01 09:25:00,135.85,136.15,135.55,136.15,534765,0


Pandas dtype:
datetime     object
open        float64
high        float64
low         float64
close       float64
volume        int64
oi            int64
dtype: object

Parquet schema:
datetime: string
open: double
high: double
low: double
close: double
volume: int64
oi: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 855


In [82]:
# Cell 8: Download Full 4 Years - TATASTEEL
# ⚠️ WARNING: This will take ~2-3 minutes

START_YEAR = 2022
END_YEAR = 2025

print("🚀 Starting full download...")
print(f"   Period: {START_YEAR}-{END_YEAR} (4 years)")
print("   This will take ~2-3 minutes\n")

all_data = []
current_date = datetime(START_YEAR, 1, 1)
end_date = datetime(END_YEAR, 12, 31)
month_count = 0

while current_date < end_date:
    month_end = datetime(current_date.year, current_date.month, calendar.monthrange(current_date.year, current_date.month)[1])
    if month_end > end_date:
        month_end = end_date

    print(f"📅 {current_date.strftime('%Y-%m')}...", end=" ")

    df = fetch_data(TATASTEEL_KEY, current_date, month_end)

    if df is not None and len(df) > 0:
        all_data.append(df)
        print(f"✅ {len(df)} candles")
        month_count += 1
    else:
        print("⚠️  No data")

    current_date = month_end + timedelta(days=1)
    time.sleep(0.5)

print(f"\n✅ Download complete!")
print(f"📊 Processed {month_count} months")

# Combine all data
df_full = pd.concat(all_data, ignore_index=True)
df_full = df_full.drop_duplicates(subset=['datetime']).sort_values('datetime').reset_index(drop=True)

print(f"📦 Total candles: {len(df_full):,}")



🚀 Starting full download...
   Period: 2022-2025 (4 years)
   This will take ~2-3 minutes

📅 2022-01... ✅ 1500 candles
📅 2022-02... ✅ 1500 candles
📅 2022-03... ✅ 1574 candles
📅 2022-04... ✅ 1425 candles
📅 2022-05... ✅ 1575 candles
📅 2022-06... ✅ 1650 candles
📅 2022-07... ✅ 1575 candles
📅 2022-08... ✅ 1500 candles
📅 2022-09... ✅ 1650 candles
📅 2022-10... ✅ 1362 candles
📅 2022-11... ✅ 1575 candles
📅 2022-12... ✅ 1650 candles
📅 2023-01... ✅ 1575 candles
📅 2023-02... ✅ 1500 candles
📅 2023-03... ✅ 1575 candles
📅 2023-04... ✅ 1275 candles
📅 2023-05... ✅ 1650 candles
📅 2023-06... ✅ 1575 candles
📅 2023-07... ✅ 1575 candles
📅 2023-08... ✅ 1650 candles
📅 2023-09... 

KeyboardInterrupt: 

In [74]:
# Cell 9: Validate Full Dataset

print("🔍 Dataset Validation\n")

# Date range
print(f"📅 Date Range:")
print(f"   Start: {df_full['datetime'].min()}")
print(f"   End:   {df_full['datetime'].max()}")

# Trading days
years = df_full.groupby(df_full['datetime'].dt.year).size()
print(f"\n📊 Candles per Year:")
display(years)

# Price range
print(f"\n💹 Price Statistics:")
print(f"   Highest: ₹{df_full['high'].max():.2f}")
print(f"   Lowest:  ₹{df_full['low'].min():.2f}")
print(f"   Latest:  ₹{df_full['close'].iloc[-1]:.2f}")

# Save to parquet
output_path = Path("../data/historical/intraday_5min/TATASTEEL.parquet")
df_full.to_parquet(output_path, index=False)
print(f"\n💾 Saved final dataset:")
print(f"   Path: {output_path}")
print(f"   Size: {output_path.stat().st_size / (1024*1024):.1f} MB")

🔍 Dataset Validation

📅 Date Range:
   Start: 2022-01-03 09:15:00+05:30
   End:   2025-12-31 15:25:00+05:30

📊 Candles per Year:


datetime
2022    18536
2023    18387
2024    18504
2025    18612
dtype: int64


💹 Price Statistics:
   Highest: ₹186.94
   Lowest:  ₹79.10
   Latest:  ₹179.93

💾 Saved final dataset:
   Path: ..\data\historical\intraday_5min\TATASTEEL.parquet
   Size: 1.7 MB


In [75]:
# Cell 10: Summary & Next Steps

print("="*70)
print("✅ DATA DOWNLOAD EXPLORATION COMPLETE")
print("="*70)
print(f"\n📊 What we learned:")
print(f"   1. Upstox API returns OHLCV data in 5-min intervals")
print(f"   2. ~75 candles per trading day (9:15 AM - 3:30 PM)")
print(f"   3. Data saved as parquet (fast loading, small size)")
print(f"   4. TATASTEEL: {len(df_full):,} candles over 4 years")

print(f"\n📁 File saved: ../data/historical/intraday_5min/TATASTEEL.parquet")

print(f"\n🎯 Next Steps:")
print(f"   1. Run download_data.py to get remaining 30 stocks")
print(f"   2. Build 02_indicator_calculation.ipynb")
print(f"   3. Build 03_bounce_detection.ipynb")

✅ DATA DOWNLOAD EXPLORATION COMPLETE

📊 What we learned:
   1. Upstox API returns OHLCV data in 5-min intervals
   2. ~75 candles per trading day (9:15 AM - 3:30 PM)
   3. Data saved as parquet (fast loading, small size)
   4. TATASTEEL: 74,039 candles over 4 years

📁 File saved: ../data/historical/intraday_5min/TATASTEEL.parquet

🎯 Next Steps:
   1. Run download_data.py to get remaining 30 stocks
   2. Build 02_indicator_calculation.ipynb
   3. Build 03_bounce_detection.ipynb
